In [ ]:
!pip install ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import ftfy
from sklearn.model_selection import train_test_split

# ─── LOAD ──────────────────────────────────────────────

df = pd.read_csv("reddit_data.csv", encoding="utf-8")
print("Columns:", df.columns.tolist())
print("Size before cleaning:", len(df))
print("Label distribution:\n", df["label"].value_counts())

# ─── CLEAN ─────────────────────────────────────────────

clean = df[["comment", "parent_comment", "label"]].copy()
clean.columns = ["reply", "parent", "label"]
clean["source"] = "reddit"

# Fix encoding (mojibake → proper text + emojis)
clean["reply"]  = clean["reply"].apply(lambda x: ftfy.fix_text(str(x)))
clean["parent"] = clean["parent"].apply(lambda x: ftfy.fix_text(str(x)))

# Remove empty/broken rows
clean = clean.dropna(subset=["reply", "parent", "label"])
clean = clean[clean["reply"].str.strip()  != ""]
clean = clean[clean["parent"].str.strip() != ""]
clean = clean[clean["label"].isin([0, 1])]

print("Size after cleaning:", len(clean))

# ─── SAMPLING — 10k sarcastic + 10k non-sarcastic ──────

sarc     = clean[clean["label"] == 1].sample(n=10000, random_state=42)
non_sarc = clean[clean["label"] == 0].sample(n=10000, random_state=42)
balanced = pd.concat([sarc, non_sarc]).sample(frac=1, random_state=42)

print("Final size:", len(balanced))
print("Label distribution:\n", balanced["label"].value_counts())

# ─── SPLIT 80 / 10 / 10 ────────────────────────────────

train, temp = train_test_split(balanced, test_size=0.2, random_state=42,
                               stratify=balanced["label"])
val, test   = train_test_split(temp, test_size=0.5, random_state=42,
                               stratify=temp["label"])

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

# ─── SAVE ──────────────────────────────────────────────

train.to_csv("train.csv", index=False)
val.to_csv(  "val.csv",   index=False)
test.to_csv( "test.csv",  index=False)

Columns: ['label', 'comment', 'author', 'subreddit', 'score', 'ups', 'downs', 'date', 'created_utc', 'parent_comment']
Size before cleaning: 1010826
Label distribution:
 label
0    505413
1    505413
Name: count, dtype: int64
Size after cleaning: 1010826
Final size: 20000
Label distribution:
 label
0    10000
1    10000
Name: count, dtype: int64
Train: 16000 | Val: 2000 | Test: 2000
